# Laser Off Code

In [ ]:
############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

# Import

In [1]:
import time
from time import sleep, monotonic
import datetime
import numpy as np
import matplotlib.pyplot as plt
import sys
import pyvisa
import qcodes as qc
from qcodes.dataset import Measurement
from qcodes.dataset import do0d
from qcodes.dataset.experiment_container import new_experiment, load_experiment_by_name
from qcodes.dataset.plotting import plot_by_id
from qcodes.dataset.data_set import load_by_id, load_by_counter
from qcodes import initialise_or_create_database_at, new_data_set, new_experiment
from qcodes.station import Station
initialise_or_create_database_at("./2026-04-17_SNSPD5.db")
from functions import quick_check
from functions import calibrate
import snspd
params = snspd.snspd()

# Set up experiment
exp_name = 'SNSPD5'
sample_name = '00'

try:
    exp = qc.load_experiment_by_name(exp_name, sample=sample_name)
    print('Experiment loaded. Last ID no:', exp.last_counter)
except ValueError:
    exp = new_experiment(exp_name, sample_name)
    print('Started new experiment')

Logging hadn't been started.
Activating auto-logging. Current session state plus future input saved.
Filename       : C:\Users\QNL\.qcodes\logs\command_history.log
Mode           : append
Output logging : True
Raw input log  : False
Timestamping   : True
State          : active
Qcodes Logfile : C:\Users\QNL\.qcodes\logs\260418-34072-qcodes.log
Experiment loaded. Last ID no: 9


# Calibration


Expected value power10 with PM100USB downstream of the 10% port and PM120 at the 90% port. 

$$ \begin{align*}
    power90 & = bs90*plaser \\
    plaser & = \frac{power90}{bs90} \\
    power10 & < bs10*plaser\\
    power10 & < bs10*\frac{power90}{bs90} \\
    \end{align*} $$

In [37]:
for ID in range(1,5): 
    data = load_by_id(ID).get_parameter_data()
    timestamp = load_by_id(ID).run_timestamp()
    connection = load_by_id(ID).metadata['attenuator_name']
    power10 = data['power10']['power10']
    power90 = data['power90']['power90']
    print(ID)
    print(connection)
    print(f'power90: {power90}')
    print(f'power10: {power10}')
    expected = params.bs10*power90/params.bs90
    print(f'Expected max power10: {expected}')
    print(f'Condition met? {'Yes' if power90 < expected else 'No'}')
    print('\n')

1
single fibre
power90: [0.00483268]
power10: [0.0021353]
Expected max power10: [0.00046743]
Condition met? No


2
single fibre
power90: [0.00480654]
power10: [0.00212446]
Expected max power10: [0.0004649]
Condition met? No


3
single fibre
power90: [0.00481491]
power10: [0.00212648]
Expected max power10: [0.00046571]
Condition met? No


4
single fibre
power90: [0.00482118]
power10: [0.00200799]
Expected max power10: [0.00046632]
Condition met? No




Expected value of power90 with PM120 downstream of the 10% port and PM100USB disconnected. Variables swap positions because variables were not renamed when power meters were swapped. Since PM100USB has a maximum power limit of 3mW, it cannot measure 4.8mW at the 90% port after the laser. The laser cannot be set lower than 7mW. Taking the value 4.8mW at the 90% port to be equal to the variable power10, the expected value of power90 becomes: 

$$ \begin{align*}
    power90 & < bs10*\frac{0.00482118}{bs90} \\
    \end{align*} $$

In [35]:
for ID in range(6,7): 
    data = load_by_id(ID).get_parameter_data()
    timestamp = load_by_id(ID).run_timestamp()
    connection = load_by_id(ID).metadata['attenuator_name']
    power10 = data['power10']['power10']
    power90 = data['power90']['power90']
    print(ID)
    print(connection)
    power_90pc_port = 0.00482118
    print(f'power10: {power10} (should be small, PM100USB disconnected)')
    print(f'power90: {power90}')
    print(f'Power at 90% port: {power_90pc_port}')
    expected = params.bs10*power_90pc_port/params.bs90
    print(f'Expected max power90: {expected}')
    print(f'Condition met? {'Yes' if power90 < expected else 'No'}')
    print('\n')

6
single fibre
power10: [1.42649972e-11] (should be small, PM100USB disconnected)
power90: [0.00038278]
Power at 90% port: 0.00482118
Expected max power90: 0.00046631527958688297
Condition met? Yes




In [36]:
for ID in range(7,10): 
    data = load_by_id(ID).get_parameter_data()
    timestamp = load_by_id(ID).run_timestamp()
    connection = load_by_id(ID).metadata['attenuator_name']
    power10 = data['power10']['power10']
    power90 = data['power90']['power90']
    print(ID)
    print(connection)
    power_90pc_port = 0.00482118
    print(f'power10: {power10} (should be small, PM100USB disconnected)')
    print(f'power90: {power90}')
    print(f'Power at 90% port: {power_90pc_port}')
    expected = params.bs10*power_90pc_port/params.bs90
    print(f'Expected max power90: {expected}')
    print(f'Condition met? {'Yes' if power90 < expected else 'No'}')
    print('\n')

7
ferrule to ferrule
power10: [1.99510445e-14] (should be small, PM100USB disconnected)
power90: [0.00033329]
Power at 90% port: 0.00482118
Expected max power90: 0.00046631527958688297
Condition met? Yes


8
ferrule to ferrule
power10: [1.99510445e-14] (should be small, PM100USB disconnected)
power90: [0.00033269]
Power at 90% port: 0.00482118
Expected max power90: 0.00046631527958688297
Condition met? Yes


9
ferrule to ferrule
power10: [7.14247411e-12] (should be small, PM100USB disconnected)
power90: [0.00033232]
Power at 90% port: 0.00482118
Expected max power90: 0.00046631527958688297
Condition met? Yes


